# Zero Context · TrafficWatch evidence audit

## tl;dr

**Share with caveats.** Executed top-to-bottom on 27 September 2026: both offline
passes contain exactly the same **163 segments and 33,075 per-frame risk outputs**
across four originals (18.3934 minutes). Hash, format, coverage and runtime checks pass.
First-pass runtime is 1.24–1.41× clip duration; repeat runtime is 1.37–1.69× on this host.
The revised lamp reader agrees with 45/46 selected tuning observations, versus 11/46
for the fixed-layout baseline. This is **not held-out accuracy or labelled event F1**.
The additional default whole-folder command also matches exactly (1.37–1.48× duration).

## Context & Methods

**Audience:** reviewers and teammates checking the claims in the technical report.
**Question:** do sample counts, runtimes, repeated outputs and the scoped lamp review
support the claims made about this model revision?

Sources are the committed `reports/original_gpu/calibrated/` first/repeat runs and the additional one-process whole-folder run,
their manifests and frozen inference hashes; `data/labels/signal_spotchecks.json`;
and `reports/scene_review/signal_review.json`. All recorded work is from 27 September 2026.
Clip time is seconds from the first frame; manifest recording times are UTC.

### Key Assumptions

- The official evaluator is unchanged; both passes use the same four original inputs.
- Timing means Part A + Part B as measured by the official harness on an RTX 3050 Laptop.
  It is not a T4 benchmark; Linux monotonic timing excludes the documented host suspend.
- Risk outputs are heuristic scores, not empirically calibrated probabilities.
- Lamp annotations are an AI-assisted **tuning** review: selected red/green frames,
  two correlated heads per frame, no amber/transition validation and no held-out data.
- Original-video bytes are not rehashed here. Input provenance is checked between saved
  run manifests; `tools/record_run.py` performs the actual input hashing.
- No model inference, hosted API calls or changes to the submission are made by this notebook.

Reproduction: Python 3.11, NumPy 2.2.6, nbformat 5.11.1, nbclient 0.11.0 and ipykernel 7.3.0.
Run from this repository or its `notebooks/` directory. Notebook execution tooling is
optional authoring tooling, not an inference dependency.

## Data

### 1. Load bounded, versioned evidence

In [1]:
from collections import Counter
import hashlib
import json
import math
from pathlib import Path
import sys
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "solution.py").is_file())
sys.path.insert(0, str(ROOT))
from evaluate import OFFICIAL_CLASSES, validate

def read_json(relative):
    return json.loads((ROOT / relative).read_text())

def sha256(relative):
    with (ROOT / relative).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n| " + " | ".join("---" for _ in headers)
                     + " |\n" + "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))

BASE = Path("reports/original_gpu/calibrated")
predictions = {stage: read_json(BASE / stage / "predictions.json") for stage in ("first", "repeat")}
manifests = {stage: read_json(BASE / stage / "run.json") for stage in ("first", "repeat")}
frozen = read_json(BASE / "frozen.json")
names = sorted(predictions["first"]["videos"])
assert set(names) == {"C3896.MP4", "C3897.MP4", "C3902.MP4", "C3905.MP4"}
print("Evidence:", len(names), "original clips; stages:", ", ".join(predictions))

Evidence: 4 original clips; stages: first, repeat


### 2. Validate identity, format and frame coverage

Primary keys: video name for outputs; `(video, frame index)` for causal risk samples.
Same-class overlap is forbidden; different classes may overlap. Event counts are
segment counts, not unique vehicles, incidents or correct detections.

In [2]:
for stage, prediction in predictions.items():
    errors, warnings = validate(prediction)
    assert not errors and not warnings, (stage, errors, warnings)
    manifest = manifests[stage]
    assert sha256(BASE / stage / "predictions.json") == manifest["predictions_sha256"]
    assert set(prediction["videos"]) == set(manifest["inputs"]) == set(names)
    for name, entry in prediction["videos"].items():
        source = manifest["inputs"][name]
        assert len(entry["events"]) == len({tuple(event) for event in entry["events"]})
        assert all(0 <= start < end <= source["duration"] and label in OFFICIAL_CLASSES
                   for start, end, label in entry["events"])
        assert len(entry["risk"]) == source["n_frames"]
        for frame, (timestamp, score) in enumerate(entry["risk"]):
            assert math.isfinite(timestamp) and math.isfinite(score) and 0 <= score <= 1
            assert abs(timestamp - frame / source["fps"]) <= 0.000051
        assert all(a[0] < b[0] for a, b in zip(entry["risk"], entry["risk"][1:]))
        log = prediction["log"][name]
        assert not log["errors"] and log["total_sec"] <= log["budget_sec"]

for field in ("source_sha256", "config_sha256", "inputs", "packages", "python", "profile"):
    assert manifests["first"][field] == manifests["repeat"][field], field
for relative, expected in frozen.items():
    assert sha256(relative) == expected, relative
assert predictions["first"]["videos"] == predictions["repeat"]["videos"]
assert len({item["sha256"] for item in manifests["first"]["inputs"].values()}) == len(names)
print("PASS: hashes, official format, finite per-frame risk, valid event bounds, budgets and exact repeat equality.")

PASS: hashes, official format, finite per-frame risk, valid event bounds, budgets and exact repeat equality.


## Results

### 3. Recompute counts and runtime denominators

Runtime ratio = `(Part A + Part B seconds) / (input frames / input FPS)`.
The table retains both runs instead of selecting only the fastest measurements.

In [3]:
rows = []
total_events = total_risk = 0
total_duration = 0.0
first_ratios, repeat_ratios = [], []
for name in names:
    entry = predictions["first"]["videos"][name]
    source = manifests["first"]["inputs"][name]
    duration = source["n_frames"] / source["fps"]
    assert abs(duration - source["duration"]) < 1e-8
    first_time = predictions["first"]["log"][name]["total_sec"]
    repeat_time = predictions["repeat"]["log"][name]["total_sec"]
    first_ratios.append(first_time / duration)
    repeat_ratios.append(repeat_time / duration)
    total_events += len(entry["events"])
    total_risk += len(entry["risk"])
    total_duration += duration
    rows.append([name, len(entry["events"]), len(entry["risk"]), f"{duration:.2f}",
                 f"{first_time:.1f}", f"{repeat_time:.1f}", f"{first_time/duration:.2f}×", f"{repeat_time/duration:.2f}×"])
table(["Original", "Segments", "Risk frames", "Duration s", "First s", "Repeat s", "First ratio", "Repeat ratio"], rows)
print(f"Total: {total_events} segments; {total_risk:,} risk frames; {total_duration/60:.4f} minutes.")
print(f"First-pass range: {min(first_ratios):.2f}–{max(first_ratios):.2f}×; repeat range: {min(repeat_ratios):.2f}–{max(repeat_ratios):.2f}×.")

| Original | Segments | Risk frames | Duration s | First s | Repeat s | First ratio | Repeat ratio |
| --- | --- | --- | --- | --- | --- | --- | --- |
| C3896.MP4 | 41 | 10200 | 340.34 | 452.1 | 574.4 | 1.33× | 1.69× |
| C3897.MP4 | 54 | 9525 | 317.82 | 436.4 | 434.5 | 1.37× | 1.37× |
| C3902.MP4 | 48 | 9525 | 317.82 | 449.7 | 516.6 | 1.41× | 1.63× |
| C3905.MP4 | 20 | 3825 | 127.63 | 158.0 | 203.0 | 1.24× | 1.59× |

Total: 163 segments; 33,075 risk frames; 18.3934 minutes.
First-pass range: 1.24–1.41×; repeat range: 1.37–1.69×.


### 3b. Check the default whole-folder command

This additional run uses the standard command with all four videos in one fresh
process and no network access, runtime cache, profile or device override. This
checks for cross-video process state as well as the per-video repeats above.
The non-scoring team metadata differs; event/risk outputs must still match exactly.


In [4]:
batch = read_json(BASE / "batch/predictions.json")
batch_manifest = read_json(BASE / "batch/run.json")
comparison = read_json(BASE / "batch/comparison.json")
errors, warnings = validate(batch)
assert not errors and not warnings
assert sha256(BASE / "batch/predictions.json") == batch_manifest["predictions_sha256"]
assert comparison["batch_predictions_sha256"] == batch_manifest["predictions_sha256"]
assert comparison["reference_predictions_sha256"] == manifests["first"]["predictions_sha256"]
assert comparison["exact_events_and_risk"] and comparison["network_interfaces"] == ["lo"]
assert batch["videos"] == predictions["first"]["videos"]
for field in ("source_sha256", "config_sha256", "inputs", "packages", "python", "profile"):
    assert batch_manifest[field] == manifests["first"][field], field
batch_rows, batch_ratios = [], []
for name in names:
    log = batch["log"][name]
    assert not log["errors"] and log["total_sec"] <= log["budget_sec"]
    ratio = log["total_sec"] / batch_manifest["inputs"][name]["duration"]
    batch_ratios.append(ratio)
    batch_rows.append([name, log["total_sec"], f"{ratio:.2f}×", "exact event/risk match"])
table(["Original", "Whole-folder s", "Runtime / duration", "Comparison"], batch_rows)
print(f"PASS: one-process folder run; runtime range {min(batch_ratios):.2f}–{max(batch_ratios):.2f}×.")


| Original | Whole-folder s | Runtime / duration | Comparison |
| --- | --- | --- | --- |
| C3896.MP4 | 487.2 | 1.43× | exact event/risk match |
| C3897.MP4 | 463.4 | 1.46× | exact event/risk match |
| C3902.MP4 | 470.0 | 1.48× | exact event/risk match |
| C3905.MP4 | 174.9 | 1.37× | exact event/risk match |

PASS: one-process folder run; runtime range 1.37–1.48×.


### 4. Compare emitted segment counts, not accuracy

The preceding accident-rule revision is a development baseline on the same originals.
Registration and lamp reading changed; no independent event ground truth is available.
A lower or higher count is **not** evidence of better precision, recall or macro F1.

In [5]:
baseline = read_json("reports/original_gpu/predictions-reviewed.json")
before = Counter(event[2] for entry in baseline["videos"].values() for event in entry["events"])
after = Counter(event[2] for entry in predictions["first"]["videos"].values() for event in entry["events"])
table(["Official label", "Previous segments", "Registered/hue segments"],
      [[label, before[label], after[label]] for label in OFFICIAL_CLASSES])
assert sum(after.values()) == total_events
print("Event precision / recall / F1 and accident anticipation score: NOT MEASURED (no independent event labels).")

| Official label | Previous segments | Registered/hue segments |
| --- | --- | --- |
| accident | 9 | 9 |
| near_miss | 0 | 0 |
| red_light | 3 | 6 |
| wrong_way | 0 | 0 |
| illegal_u_turn | 0 | 0 |
| stopped_vehicle | 14 | 18 |
| jaywalking | 34 | 33 |
| failure_to_yield | 96 | 91 |
| illegal_turn | 0 | 0 |
| solid_line_crossing | 0 | 0 |
| stop_line | 2 | 4 |
| congestion | 7 | 2 |
| road_obstacle | 0 | 0 |
| fire_smoke | 0 | 0 |

Event precision / recall / F1 and accident anticipation score: NOT MEASURED (no independent event labels).


### 5. Reconcile the scoped lamp review

Primary key: `(original video, decoded frame index, lamp head)`. The denominator includes
every selected lamp, including `unknown` reader outputs. This checks agreement with
the tuning review, not hidden-test accuracy; paired heads are not independent samples.

In [6]:
labels = read_json("data/labels/signal_spotchecks.json")
review = read_json("reports/scene_review/signal_review.json")
assert review["scope"] == labels["scope"]
expected = {}
for video, samples in labels["videos"].items():
    source = manifests["first"]["inputs"][video]
    for timestamp, state in samples:
        assert 0 <= timestamp < source["duration"] and state in {"red", "green"}
        for light in labels["lights"]:
            key = (video, round(timestamp * source["fps"]), light)
            assert key not in expected
            expected[key] = state

observations = review["observations"]
keys = [(row["video"], row["frame"], row["light"]) for row in observations]
assert len(keys) == len(set(keys)) and set(keys) == set(expected)
modes = ("fixed_layout_brightness", "registered_brightness", "registered_hue")
for row, key in zip(observations, keys):
    assert row["reviewed_state"] == expected[key]
    fps = manifests["first"]["inputs"][row["video"]]["fps"]
    assert abs(row["time"] - row["frame"] / fps) < 1e-9
    assert all(row[mode] in {"red", "amber", "green", "unknown"} for mode in modes)

counts = {mode: {"matches_review": sum(row[mode] == row["reviewed_state"] for row in observations),
                 "unknown": sum(row[mode] == "unknown" for row in observations),
                 "total": len(observations)} for mode in modes}
assert counts == review["counts"]
table(["Reader variant", "Matches / selected lamps", "Agreement", "Unknown (included)"],
      [[mode, f"{value['matches_review']} / {value['total']}",
        f"{100*value['matches_review']/value['total']:.1f}%", value["unknown"]]
       for mode, value in counts.items()])
print("Coverage by original:", dict(Counter(row["video"] for row in observations)))
print("Reviewed state mix:", dict(Counter(row["reviewed_state"] for row in observations)))
print(labels["scope"])

| Reader variant | Matches / selected lamps | Agreement | Unknown (included) |
| --- | --- | --- | --- |
| fixed_layout_brightness | 11 / 46 | 23.9% | 29 |
| registered_brightness | 27 / 46 | 58.7% | 18 |
| registered_hue | 45 / 46 | 97.8% | 1 |

Coverage by original: {'C3896.MP4': 12, 'C3897.MP4': 12, 'C3902.MP4': 12, 'C3905.MP4': 10}
Reviewed state mix: {'red': 26, 'green': 20}
AI-assisted visual review of 23 sampled frames / 46 visible lamp states. Not independently annotated, held out, or event ground truth.


## Takeaways

- The recorded runs, including the default whole-folder command, support sample
  coverage, same-host exact repeatability and the
  observed runtime claims. No missing risk frames, duplicate event tuples, invalid
  timestamps or source/configuration hash differences were found.
- The staged lamp comparison is 11/46 → 27/46 → 45/46 matches. Unknown readings
  remain in the denominator: 29 → 18 → 1. The 46 observations cover 23 selected
  frames, with 26 red and 20 green states; correlated heads are not independent trials.
- Event counts changed from 165 to 163; this cannot establish better precision,
  recall or F1. Nine accident candidates remain unconfirmed.
- Confidence is high in these arithmetic/provenance checks, but generalisation
  cannot be inferred from this tuning sample. Independent event labels, transition
  review and a T4 run are the next evidence needed.

Unresolved handoff blockers (separate from analytical caveats): public release is not
authorised yet; public hosting and a real public upload remain unverified; profile links
and previous projects are incomplete. Lack of a T4 benchmark, independent event labels
and empirical risk calibration limits claims, even when package/browser checks pass.